# 04. Dataset Improvements

This notebook applies the cleanup decisions that came from the first full dataset build and the EDA.

The goal here is not to rebuild the dataset from scratch. The goal is to make the finished catalog cleaner before the Week 3, Week 5, and Week 7 notebooks use it:

- remove the tiny group of entries with no usable air date
- fill missing or zero episode counts from the AniDB cache, with an optional live AniDB repair cell
- infer missing MAL genres from tags when the tag name clearly matches a known genre
- fill empty studio values with AniDB `origin` tags when studio metadata is unavailable
- refill missing tags and demographics from AniDB cache metadata

Live AniDB calls are separated into their own cell so a normal run stays cache-first and does not trigger unexpected requests.


## Setup

The implementation lives in `src/04_apply_dataset_improvements.py` so this same cleanup can be rerun from a command:

```bash
python src/04_apply_dataset_improvements.py
```

This notebook imports the script functions, previews what will change, and then saves the improved `anime_dataset.csv` and `anime_dataset.json`.


In [59]:
from pathlib import Path
import sys
import importlib.util

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

MODULE_PATH = ROOT / "src" / "04_apply_dataset_improvements.py"
spec = importlib.util.spec_from_file_location("dataset_improvements", MODULE_PATH)
improve = importlib.util.module_from_spec(spec)
spec.loader.exec_module(improve)

DATASET_CSV = ROOT / "data" / "processed" / "anime_dataset.csv"
DATASET_JSON = ROOT / "data" / "processed" / "anime_dataset.json"
ANIDB_CACHE = ROOT / "data" / "caches" / "anidb_metadata_cache.json"

df = pd.read_csv(DATASET_CSV)
cache_payload = improve.load_anidb_cache()

print(f"Dataset rows: {len(df):,}")
print(f"AniDB cache entries: {len(cache_payload.get('items', {})):,}")


Dataset rows: 16,664
AniDB cache entries: 15,719


## Baseline Null Audit

This audit treats real nulls, empty strings, and string placeholders such as `null` as missing. That matters for this dataset because values came from multiple APIs and file formats.


In [60]:
audit_columns = [
    "aired_year",
    "aired_month",
    "episodes",
    "season",
    "duration",
    "total_watch_minutes",
    "genres",
    "studios",
    "tags",
    "explicit_tags",
    "demographics",
]

missing_rows = []
for column in audit_columns:
    if column in {"aired_year", "aired_month", "episodes", "duration", "total_watch_minutes"}:
        missing = df[column].isna()
        if column == "episodes":
            missing = missing | (pd.to_numeric(df[column], errors="coerce").fillna(-1) == 0)
    elif column == "explicit_tags":
        missing = df[column].apply(improve.is_missing_text) & df.apply(improve.should_have_explicit_tags, axis=1)
    else:
        missing = df[column].apply(improve.is_missing_text)
    missing_rows.append({"column": column, "missing_or_zero": int(missing.sum())})

pd.DataFrame(missing_rows)


,column,missing_or_zero
0,aired_year,0
1,aired_month,0
2,episodes,0
3,season,0
4,duration,22
5,total_watch_minutes,22
6,genres,588
7,studios,1148
8,tags,2715
9,explicit_tags,228


## Rule 1: Remove Entries With No Air Date

Rows with both `aired_year` and `aired_month` missing are removed. There are very few of them, and without a date they are weak for temporal features, season inference, and milestone reporting.


In [61]:
missing_air_date = df["aired_year"].isna() & df["aired_month"].isna()
df.loc[
    missing_air_date,
    ["mal_id", "title", "type", "score", "members", "aired_year", "aired_month"],
].sort_values("mal_id")


,mal_id,title,type,score,members,aired_year,aired_month


## Rule 1b: Fill Season From Air Month

`season` is redundant with `aired_month` when the month exists, so empty season values are filled deterministically:

- January to March: `winter`
- April to June: `spring`
- July to September: `summer`
- October to December: `fall`


In [62]:
season_preview = df[["mal_id", "title", "aired_month", "season"]].copy()
season_preview["season_from_month"] = season_preview["aired_month"].apply(improve.infer_season)

season_preview[
    season_preview["season"].apply(improve.is_missing_text)
    & season_preview["season_from_month"].notna()
].head(25)


,mal_id,title,aired_month,season,season_from_month


## Rule 1c: Convert Duration to Minutes and Add Total Watch Time

The raw duration text is useful for inspection, but numeric modeling needs minutes. This step replaces `duration` with numeric minutes and adds:

`total_watch_minutes = episodes * duration`

Unknown durations stay null.


In [63]:
runtime_preview = df[["mal_id", "title", "episodes", "duration"]].copy()
runtime_preview["duration_minutes"] = runtime_preview["duration"].apply(improve.parse_duration_minutes)
runtime_preview["total_watch_minutes"] = (
    pd.to_numeric(runtime_preview["episodes"], errors="coerce")
    * pd.to_numeric(runtime_preview["duration_minutes"], errors="coerce")
)

runtime_preview.head(25)


,mal_id,title,episodes,duration,duration_minutes,total_watch_minutes
0,1,Cowboy Bebop,26.0,24.0,24.0,624.0
1,5,Cowboy Bebop: Tengoku no Tobira,1.0,115.0,115.0,115.0
2,6,Trigun,26.0,24.0,24.0,624.0
3,7,Witch Hunter Robin,26.0,25.0,25.0,650.0
4,8,Bouken Ou Beet,52.0,23.0,23.0,1196.0
5,15,Eyeshield 21,145.0,23.0,23.0,3335.0
6,16,Hachimitsu to Clover,24.0,23.0,23.0,552.0
7,17,Hungry Heart: Wild Striker,52.0,23.0,23.0,1196.0
8,18,Initial D Fourth Stage,24.0,27.0,27.0,648.0
9,19,Monster,74.0,24.0,24.0,1776.0


## Rule 2: Fill Missing Episode Counts From AniDB

MAL sometimes has `episodes = null` or `episodes = 0` for currently airing or recently announced titles. AniDB can fill some of these from the local cache. Anything still missing after the cache pass becomes a live-call candidate.


In [64]:
episode_gap = df["episodes"].isna() | (pd.to_numeric(df["episodes"], errors="coerce").fillna(-1) == 0)
episode_preview = df.loc[
    episode_gap,
    ["mal_id", "anidb_id", "title", "type", "status", "episodes", "popularity"],
].copy()

items = cache_payload.get("items", {})
episode_preview["cached_episode_count"] = episode_preview["anidb_id"].apply(
    lambda value: (
        items.get(str(int(value)), {}).get("episode_count")
        if pd.notna(value)
        else None
    )
)
episode_preview.sort_values(["cached_episode_count", "popularity"], ascending=[False, True])


,mal_id,anidb_id,title,type,status,episodes,popularity,cached_episode_count


## Optional Live AniDB Repair

Run these cells only when you are ready to make live AniDB HTTP calls. They update the local AniDB cache first, then the normal improvement step uses the refreshed cache.

The order matters because live AniDB requests are valuable:

1. repair missing/zero episode counts first
2. then repair tags, explicit tags, and demographics by MAL popularity

Each successful AniDB response is saved immediately to `data/caches/anidb_metadata_cache.json`.


In [65]:
# Step 1: episode repair candidates only.
# This catches cached AniDB episode counts that are missing OR still zero.

episode_live_candidates = improve.episode_live_candidate_frame(df, cache_payload)
episode_candidate_ids = episode_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live episode candidates: {len(episode_candidate_ids)}")
display(episode_live_candidates)

# Uncomment to spend live AniDB calls on episode gaps.
updated = improve.update_cache_with_live_payloads(
     cache_payload,
     episode_candidate_ids,
     label="episode_repair",
 )
print(f"Live AniDB episode cache updates: {updated}")


Live episode candidates: 0


,mal_id,anidb_id,title,type,status,episodes,popularity,cached_episode_count


Live AniDB episode cache updates: 0


### Optional Live AniDB Duration Repair

Run this after episode repair. It targets rows where `duration` or `total_watch_minutes` is still missing and sorts by MAL popularity.


In [66]:
# Step 2: duration / total watch-time repair candidates.

duration_live_candidates = improve.duration_live_candidate_frame(df)
duration_candidate_ids = duration_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live duration/runtime candidates: {len(duration_candidate_ids)}")
display(duration_live_candidates.head(100))

# Uncomment to spend live AniDB calls on duration/runtime gaps.
#updated = improve.update_cache_with_live_payloads(
#     cache_payload,
#     duration_candidate_ids,
#     label="duration_runtime_repair",
# )
#print(f"Live AniDB duration/runtime cache updates: {updated}")


Live duration/runtime candidates: 15


,mal_id,anidb_id,title,type,status,episodes,duration,total_watch_minutes,popularity,needs_duration,needs_total_watch_minutes
14287,63074,19756.0,"Mou Ichido, Shitemitai.",OVA,Currently Airing,1.0,NaN,NaN,12932,True,True
14280,62578,19611.0,Cool de M,OVA,Currently Airing,2.0,NaN,NaN,13824,True,True
4617,10742,5384.0,Saru to Kani no Gassen,Movie,Finished Airing,1.0,NaN,NaN,14030,True,True
14290,63158,19769.0,Hebi to Kumo,OVA,Currently Airing,1.0,NaN,NaN,14146,True,True
14291,63159,19762.0,Anal Mania Otaku to Ananii Daisuki na Ojou-sama,OVA,Currently Airing,2.0,NaN,NaN,14286,True,True
14279,62577,19612.0,Seihou Shouka Saint Lime,OVA,Currently Airing,2.0,NaN,NaN,14351,True,True
14285,63026,19738.0,Kenki Virgo,OVA,Currently Airing,2.0,NaN,NaN,14505,True,True
14286,63027,19737.0,L'amour fou de l'automate,OVA,Currently Airing,2.0,NaN,NaN,14810,True,True
14292,63232,19790.0,Phantom Alchemia: Silvia no Dokidoki Sakusei T...,OVA,Currently Airing,2.0,NaN,NaN,14985,True,True
4621,10758,5386.0,Momotarou,Movie,Finished Airing,1.0,NaN,NaN,15780,True,True


[1/15] duration_runtime_repair | AniDB 19756 | request_start
[1/15] duration_runtime_repair | AniDB 19756 | no_update
[2/15] duration_runtime_repair | AniDB 19611 | request_start
[2/15] duration_runtime_repair | AniDB 19611 | no_update
[3/15] duration_runtime_repair | AniDB 5384 | request_start
[3/15] duration_runtime_repair | AniDB 5384 | no_update
[4/15] duration_runtime_repair | AniDB 19769 | request_start
[4/15] duration_runtime_repair | AniDB 19769 | no_update
[5/15] duration_runtime_repair | AniDB 19762 | request_start
[5/15] duration_runtime_repair | AniDB 19762 | no_update
[6/15] duration_runtime_repair | AniDB 19612 | request_start
[6/15] duration_runtime_repair | AniDB 19612 | no_update
[7/15] duration_runtime_repair | AniDB 19738 | request_start
[7/15] duration_runtime_repair | AniDB 19738 | no_update
[8/15] duration_runtime_repair | AniDB 19737 | request_start
[8/15] duration_runtime_repair | AniDB 19737 | no_update
[9/15] duration_runtime_repair | AniDB 19790 | request_sta

### Optional Live AniDB Currently-Airing Refresh

Run this when you want fresh episode counts and metadata for currently-airing titles. It is sorted by popularity because these calls are expensive.


In [67]:
# Step 3: currently-airing refresh candidates.

airing_live_candidates = improve.currently_airing_update_candidate_frame(df)
airing_candidate_ids = airing_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Currently-airing live refresh candidates: {len(airing_candidate_ids)}")
display(airing_live_candidates.head(100))

# Uncomment to refresh currently-airing AniDB payloads.
# Use a limit if you only want the most popular currently-airing titles.
#updated = improve.update_cache_with_live_payloads(
#     cache_payload,
#     airing_candidate_ids,
#     limit=100,
#     label="currently_airing_refresh",
#)
#print(f"Live AniDB currently-airing cache updates: {updated}")


Currently-airing live refresh candidates: 207


,mal_id,anidb_id,title,type,status,episodes,duration,total_watch_minutes,popularity
11,21,69.0,One Piece,TV,Currently Airing,1164.0,24.0,27936.0,17
201,235,266.0,Meitantei Conan,TV,Currently Airing,1204.0,24.0,28896.0,725
11034,51553,17305.0,Tongari Boushi no Atelier,TV,Currently Airing,13.0,23.0,299.0,997
12562,61316,19242.0,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,TV,Currently Airing,19.0,23.0,437.0,1268
12580,61469,19287.0,Steel Ball Run: JoJo no Kimyou na Bouken,ONA,Currently Airing,1.0,47.0,47.0,1371
...,...,...,...,...,...,...,...,...,...
14293,63248,19901.0,Ookii Onnanoko wa Suki desu ka?,ONA,Currently Airing,12.0,5.0,60.0,8536
12213,59176,19214.0,Mahou no Shimai Lulutto Lilly,TV,Currently Airing,12.0,23.0,276.0,8837
12728,62852,19674.0,Ghost Concert: Missing Songs,TV,Currently Airing,12.0,23.0,276.0,8855
11813,56524,18365.0,Tunshi Xingkong 4th Season,ONA,Currently Airing,175.0,21.0,3675.0,8872


### Optional Live AniDB Metadata Repair

Run this only after episode repair. These candidates need tags, explicit tags, or demographics and are sorted by MAL popularity so the highest-impact titles are repaired first.


In [68]:
# Step 4: metadata repair candidates by scarcity, then popularity.

metadata_live_candidates = improve.metadata_live_candidate_frame(df)
metadata_candidate_ids = metadata_live_candidates["anidb_id"].astype(int).drop_duplicates().tolist()

print(f"Live tag/explicit/demographic/studio metadata candidates: {len(metadata_candidate_ids)}")
display(metadata_live_candidates.head(100).sort_values(["scarcity_priority", "popularity"], na_position="last"))

# Uncomment to spend live AniDB calls on metadata gaps.
# Use a small limit if you want to avoid burning too many AniDB requests in one sitting.
# updated = improve.update_cache_with_live_payloads(
#     cache_payload,
#     metadata_candidate_ids,
#     limit=100,
#     label="metadata_repair",
# )
# print(f"Live AniDB metadata cache updates: {updated}")


Live tag/explicit/demographic/studio metadata candidates: 4954


,mal_id,anidb_id,title,type,rating,popularity,tags,explicit_tags,demographics,duration,total_watch_minutes,studios,needs_tags,needs_explicit_tags,needs_demographics,needs_duration,needs_total_watch_minutes,needs_studios,scarcity_priority,need_count
14287,63074,19756.0,"Mou Ichido, Shitemitai.",OVA,Rx - Hentai,12932,NaN,sex|nudity,18+,NaN,NaN,Seven,True,False,False,True,False,False,22,2
14280,62578,19611.0,Cool de M,OVA,Rx - Hentai,13824,high school|teacher x student|romance|school life,doggy style|sex|nudity|pornography|breasts|fem...,18+,NaN,NaN,Nur,False,False,False,True,False,False,22,1
4617,10742,5384.0,Saru to Kani no Gassen,Movie,G - All Ages,14030,NaN,NaN,NaN,NaN,NaN,Japanese production,True,False,True,True,False,False,22,3
14290,63158,19769.0,Hebi to Kumo,OVA,Rx - Hentai,14146,NaN,sex|nudity,18+,NaN,NaN,Shion,True,False,False,True,False,False,22,2
14291,63159,19762.0,Anal Mania Otaku to Ananii Daisuki na Ojou-sama,OVA,Rx - Hentai,14286,NaN,anal|sex|nudity|pornography,18+,NaN,NaN,Japanese production,True,False,False,True,False,False,22,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4920,12791,5814.0,Yankee Reppuu-tai,OVA,R+ - Mild Nudity,13853,Delinquents|delinquent|mafia|angst|adventure|s...,NaN,Shounen,51.0,306.0,Toei Animation,False,True,False,False,False,False,228,1
14289,63096,19259.0,Arisugawa Ren tte Honto wa Onna Nanda yo ne.,ONA,Rx - Hentai,14010,NaN,NaN,18+,6.0,48.0,Studio LEO,True,True,False,False,False,False,228,2
3728,7374,5308.0,100%,OVA,R+ - Mild Nudity,14085,NaN,NaN,18+,53.0,53.0,J.C.Staff,True,True,False,False,False,False,228,2
4038,8563,6687.0,Chikyuu Monogatari Telepath 2500,Movie,R+ - Mild Nudity,14115,fairy|space travel|science fiction|fantasy|spe...,NaN,18+,107.0,107.0,Tatsunoko Production,False,True,False,False,False,False,228,1


## Rule 3: Infer Missing Genres From Tags

When MAL `genres` is empty, the notebook checks the existing tag vocabulary. Only clear matches are used, such as `science fiction -> Sci-Fi`, `daily life -> Slice of Life`, and exact genre-name matches like `action`, `fantasy`, or `romance`.


In [69]:
genre_lookup = improve.build_genre_lookup(df)

genre_missing = df["genres"].apply(improve.is_missing_text)
genre_inference_preview = df.loc[
    genre_missing,
    ["mal_id", "title", "genres", "tags", "explicit_tags"],
].copy()
genre_inference_preview["inferred_genres"] = genre_inference_preview.apply(
    lambda row: improve.infer_genres_from_tags(row, genre_lookup),
    axis=1,
)

genre_inference_preview[
    genre_inference_preview["inferred_genres"].apply(lambda value: not improve.is_missing_text(value))
].head(30)


,mal_id,title,genres,tags,explicit_tags,inferred_genres


## Rule 4: Fill Empty Studios With AniDB Origin Tags

If a studio is missing, AniDB `origin` tags can still describe production provenance. These are not real animation-studio names, so they are only used as a fallback when `studios` is empty.


In [70]:
studio_missing = df["studios"].apply(improve.is_missing_text)
studio_preview = df.loc[
    studio_missing,
    ["mal_id", "anidb_id", "title", "studios"],
].copy()
studio_preview["origin_fallback"] = studio_preview["anidb_id"].apply(
    lambda value: improve.origin_tags_from_payload(
        cache_payload.get("items", {}).get(str(int(value))) if pd.notna(value) else None
    )
)
studio_preview[
    studio_preview["origin_fallback"].apply(lambda value: not improve.is_missing_text(value))
].head(30)


,mal_id,anidb_id,title,studios,origin_fallback


## Rule 5: Refill Missing Tags and Demographics

For rows where `tags`, `explicit_tags`, or demographics are empty, the notebook reclassifies the raw AniDB tags from the cache using the same rules as the dataset builder:

- taxonomy containers are ignored
- demographics come from AniDB `target audience`
- explicit tags stay separate from recommender tags and are refilled for explicit-rated rows
- tag weights are preserved


In [71]:
tag_or_demo_gap = (
    df["tags"].apply(improve.is_missing_text)
    | df["demographics"].apply(improve.is_missing_text)
    | (df["explicit_tags"].apply(improve.is_missing_text) & df.apply(improve.should_have_explicit_tags, axis=1))
)

tag_demo_preview = []
for _, row in df.loc[tag_or_demo_gap].head(40).iterrows():
    anidb_id = improve.parse_int(row.get("anidb_id"), default=None)
    payload = cache_payload.get("items", {}).get(str(anidb_id)) if anidb_id is not None else None
    parsed = improve.parse_anidb_tag_records((payload or {}).get("raw_tags", []), rating=row.get("rating"))
    tag_demo_preview.append(
        {
            "mal_id": row["mal_id"],
            "title": row["title"],
            "current_tags": row.get("tags"),
            "new_tags": parsed.get("tags"),
            "current_explicit_tags": row.get("explicit_tags"),
            "new_explicit_tags": parsed.get("explicit_tags"),
            "current_demographics": row.get("demographics"),
            "new_demographics": parsed.get("demographics"),
        }
    )

pd.DataFrame(tag_demo_preview)


,mal_id,title,current_tags,new_tags,current_explicit_tags,new_explicit_tags,current_demographics,new_demographics
0,55,Arc the Lad,bounty hunter|magic|multiple couples|action|an...,bounty hunter|magic|multiple couples|action|an...,NaN,,NaN,
1,69,Cluster Edge,Military|air force|high school|angel|gunfights...,military|air force|high school|angel|gunfights...,NaN,,NaN,
2,75,Soukyuu no Fafner: Dead Aggressor,Mecha|Military|alien|robot|human enhancement|p...,military|alien|robot|human enhancement|mecha|p...,NaN,,NaN,
3,80,Kidou Senshi Gundam,Mecha|Military|Space|extrasensory perception|r...,military|extrasensory perception|robot|swordpl...,NaN,,NaN,
4,83,Kidou Senshi Gundam: Dai 08 MS Shoutai - Mille...,Mecha|Military|robot|action|science fiction|di...,robot|mecha|action|science fiction|disaster|wa...,NaN,,NaN,
5,84,Kidou Senshi Gundam 0083: Stardust Memory,Mecha|Military|Space|robot|swordplay|space tra...,military|robot|swordplay|mecha|space travel|pi...,NaN,,NaN,
6,85,Kidou Senshi Zeta Gundam,Mecha|Military|Space|robot|swordplay|human enh...,military|robot|swordplay|human enhancement|mec...,NaN,,NaN,
7,86,Kidou Senshi Gundam ZZ,Mecha|Military|Space|child soldier|robot|colla...,military|child soldier|robot|collateral damage...,NaN,,NaN,
8,87,Kidou Senshi Gundam: Gyakushuu no Char,Mecha|Military|Space|robot|human enhancement|s...,military|robot|human enhancement|mecha|space t...,NaN,,NaN,
9,88,Kidou Senshi Gundam F91,Mecha|Military|Space|robot|collateral damage|h...,military|robot|collateral damage|human enhance...,NaN,,NaN,


## Rule 6: Conservative Demographic Inference

Demographics are the hardest remaining nulls because MAL often leaves them empty and AniDB only fills them when `target audience` tags exist.

This notebook fills only high-confidence cases:

- `Rx - Hentai`, `R+ - Mild Nudity`, `Hentai`, or `Erotica` -> `18+`
- `PG - Children` -> `Kodomo`
- exact demographic-like tags such as `shounen`, `seinen`, `shoujo`, `josei`, `kodomo`, or `mina`

It does not infer demographics from broad genres like action, comedy, romance, or fantasy because those are content categories, not audience categories.


In [72]:
demographic_gap = df["demographics"].apply(improve.is_missing_text)
demographic_preview = df.loc[
    demographic_gap,
    ["mal_id", "title", "rating", "genres", "tags", "explicit_tags", "demographics"],
].copy()
demographic_preview["inferred_demographics"] = demographic_preview.apply(
    improve.infer_demographics_from_row,
    axis=1,
)

demographic_preview[
    demographic_preview["inferred_demographics"].apply(lambda value: not improve.is_missing_text(value))
].head(40)


,mal_id,title,rating,genres,tags,explicit_tags,demographics,inferred_demographics


## Apply Improvements

This cell performs the cleanup in memory and reports how many rows or fields changed before saving.


In [73]:
improved_df, summary = improve.apply_improvements(df, cache_payload)

summary_display = {
    key: value
    for key, value in summary.items()
    if not key.startswith("remaining_")
}
summary_display


{'started_rows': 16664,
 'dropped_missing_air_date': 0,
 'seasons_filled_from_aired_month': 0,
 'duration_values_parsed_to_minutes': 16650,
 'duration_filled_from_anidb_episode_lengths': 8,
 'total_watch_minutes_filled': 16650,
 'total_watch_minutes_filled_from_anidb_episode_lengths': 0,
 'episodes_filled_from_anidb_cache': 0,
 'genres_filled_from_tags': 0,
 'studios_filled_from_origin_tags': 0,
 'recommendations_augmented_from_anidb_similar_anime': 9,
 'tags_filled_from_anidb_cache': 0,
 'explicit_tags_filled_from_anidb_cache': 1,
 'demographics_filled_from_anidb_cache': 0,
 'demographics_normalized': 0,
 'demographics_inferred_from_rating_genres_tags': 0,
 'finished_rows': 16664}

The following tables are the remaining cases that need live AniDB or manual review. They are kept as notebook output, not extra dataset files.


In [74]:
pd.DataFrame(summary["remaining_episode_live_candidates"]).head(50)


""


## Optional Manual Drop: Remaining Episode Gaps

After cache and optional live repair, any row still having `episodes` as null or zero can be removed if the remaining cases are negligible. The deletion lines are commented so a normal `Run All` does not silently drop rows.


In [75]:
episode_gap_after = (
    improved_df["episodes"].isna()
    | (pd.to_numeric(improved_df["episodes"], errors="coerce").fillna(-1) == 0)
)

episode_gap_rows = improved_df.loc[
    episode_gap_after,
    ["mal_id", "anidb_id", "title", "type", "status", "episodes", "popularity"],
].sort_values(["anidb_id", "popularity"], na_position="last")

print(f"Remaining episode gaps: {len(episode_gap_rows)}")
episode_gap_rows


Remaining episode gaps: 0


,mal_id,anidb_id,title,type,status,episodes,popularity


In [ ]:
# Manual deletion cell.
# Uncomment and run this cell only if you decide the remaining episode gaps are negligible.

#episode_gap_after = (
#     improved_df["episodes"].isna()
#     | (pd.to_numeric(improved_df["episodes"], errors="coerce").fillna(-1) == 0)
# )
#improved_df = improved_df.loc[~episode_gap_after].copy()
#print(f"Rows after dropping episode gaps: {len(improved_df):,}")


Rows after dropping episode gaps: 16,664


In [76]:
pd.DataFrame(summary["remaining_tag_demographic_or_explicit_live_candidates"]).head(50)


,mal_id,anidb_id,title,tags,explicit_tags,demographics
0,55,433.0,Arc the Lad,bounty hunter|magic|multiple couples|action|an...,NaN,
1,69,2392.0,Cluster Edge,Military|air force|high school|angel|gunfights...,NaN,
2,75,1806.0,Soukyuu no Fafner: Dead Aggressor,Mecha|Military|alien|robot|human enhancement|p...,NaN,
3,80,715.0,Kidou Senshi Gundam,Mecha|Military|Space|extrasensory perception|r...,NaN,
4,83,1425.0,Kidou Senshi Gundam: Dai 08 MS Shoutai - Mille...,Mecha|Military|robot|action|science fiction|di...,NaN,
5,84,717.0,Kidou Senshi Gundam 0083: Stardust Memory,Mecha|Military|Space|robot|swordplay|space tra...,NaN,
6,85,718.0,Kidou Senshi Zeta Gundam,Mecha|Military|Space|robot|swordplay|human enh...,NaN,
7,86,719.0,Kidou Senshi Gundam ZZ,Mecha|Military|Space|child soldier|robot|colla...,NaN,
8,87,720.0,Kidou Senshi Gundam: Gyakushuu no Char,Mecha|Military|Space|robot|human enhancement|s...,NaN,
9,88,626.0,Kidou Senshi Gundam F91,Mecha|Military|Space|robot|collateral damage|h...,NaN,


## Final Audit and Save

The final audit confirms the improved missingness profile. The save cell overwrites the processed dataset files because this notebook is the approved cleanup step after the full retry run.


In [77]:
final_missing_rows = []
for column in audit_columns:
    if column in {"aired_year", "aired_month", "episodes", "duration", "total_watch_minutes"}:
        missing = improved_df[column].isna()
        if column == "episodes":
            missing = missing | (pd.to_numeric(improved_df[column], errors="coerce").fillna(-1) == 0)
    elif column == "explicit_tags":
        missing = improved_df[column].apply(improve.is_missing_text) & improved_df.apply(improve.should_have_explicit_tags, axis=1)
    else:
        missing = improved_df[column].apply(improve.is_missing_text)
    final_missing_rows.append({"column": column, "missing_or_zero_after": int(missing.sum())})

pd.DataFrame(final_missing_rows)


,column,missing_or_zero_after
0,aired_year,0
1,aired_month,0
2,episodes,0
3,season,0
4,duration,14
5,total_watch_minutes,14
6,genres,588
7,studios,1148
8,tags,2715
9,explicit_tags,227


In [46]:
improve.save_dataset(improved_df, csv_path=DATASET_CSV, json_path=DATASET_JSON)

print(f"Saved {len(improved_df):,} rows to:")
print(DATASET_CSV)
print(DATASET_JSON)


Saved 16,664 rows to:
c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\anime_dataset.csv
c:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\processed\anime_dataset.json
